# Prepare the shared FineWeb-Edu token corpus

Select a CPU runtime in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated"
DATA = WORKSPACE / "data" / "fineweb_edu"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def _snapshot_pins(text):
    # Earlier notebooks included the editable checkout. Reinstall it from the saved commit.
    pins = []
    comment = ""
    for line in text.splitlines():
        if line.startswith("#"):
            comment = line
            continue
        if line.startswith("-e ") and re.search(r"(?i)base[-_]model[-_]research\b", line + " " + comment):
            comment = ""
            continue
        if line.strip():
            pins.append(line)
        comment = ""
    return "\n".join(sorted(pins)) + "\n"


def _installed_pins(pip):
    frozen = subprocess.check_output([*pip, "freeze", "--exclude", "base-model-research"], text=True)
    return _snapshot_pins(frozen)


def _uses_checkout(pip, repo):
    packages = subprocess.check_output(
        [*pip, "list", "--editable", "--format=json", "--disable-pip-version-check"], text=True
    )
    return any(
        package["name"] == "base-model-research" and package.get("editable_project_location") == str(repo)
        for package in json.loads(packages)
    )


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        role = "benchmark" if benchmark else "training"
        venv = workspace / "envs" / runs.name / repo.name / role
        python = venv / "bin" / "python"
        if not python.exists():
            subprocess.run([sys.executable, "-m", "venv", venv], check=True)

        pip = [python, "-m", "pip"]
        snapshot = runs / ("requirements.benchmark.freeze.txt" if benchmark else "requirements.freeze.txt")
        if snapshot.exists():
            pins = _snapshot_pins(snapshot.read_text())
            if _uses_checkout(pip, repo) and _installed_pins(pip) == pins:
                return repo, python
            requirements = venv / "recorded-requirements.txt"
            requirements.write_text(pins)
        else:
            pins = None
            requirements = repo / ("requirements.txt" if benchmark else "requirements-train.txt")
            if not requirements.exists():  # Older pinned revisions used a single requirements file.
                requirements = repo / "requirements.txt"
        subprocess.run([*pip, "install", "-r", requirements, "-e", repo], check=True)
        frozen = _installed_pins(pip)
        if pins is None:
            temporary = snapshot.with_suffix(".tmp")
            temporary.write_text(frozen)
            temporary.replace(snapshot)
        elif frozen != pins:
            raise RuntimeError(f"The installed {role} packages differ from {snapshot}.")
    return repo, python

REPO, PYTHON = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([str(PYTHON), "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)


Prepare once and reuse for every run. Default: 2,048 held-out whole documents, followed by 11,957,698,560 training targets plus one lookahead token. uint32 tokens occupy about 48 GB in decimal units, plus validation, tokenizer and caches. If preparation stops, rerun this cell. Tokenization starts over if needed. A partial Drive copy resumes from verified shards.


In [ ]:
import shutil
print("Free disk GB:", shutil.disk_usage(WORKSPACE).free / 1e9)
if not (SAVED_DATA / "manifest.json").is_file():
    if not (DATA / "manifest.json").is_file():
        run("prepare-data", "--output", DATA)
    if SAVED_DATA != DATA:
        run("copy-data", "--source", DATA, "--destination", SAVED_DATA)
print("Prepared corpus:", SAVED_DATA)
